# Feature engineering SQL prototype
Prototype leakage-safe history windows before moving stable SQL into `sql/feature_store.sql`.

In [ ]:
from config import DB_CONFIG
import psycopg2
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

In [ ]:
cur.execute("""
WITH labelled AS (
  SELECT o.order_id, c.customer_unique_id, o.order_purchase_timestamp,
         ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp, o.order_id) AS visit_number,
         LAG(o.order_purchase_timestamp) OVER (PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp, o.order_id) AS prior_purchase
  FROM public.orders o JOIN public.customers c USING (customer_id)
  WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
)
SELECT order_id, visit_number, order_purchase_timestamp::date - prior_purchase::date AS days_since_prior
FROM labelled ORDER BY customer_unique_id, order_purchase_timestamp LIMIT 20
""")
cur.fetchall()

In [ ]:
cur.close()
conn.close()